# Acehnese → Indonesian: train our model once
1. Menu **Runtime → Change runtime type → T4 GPU → Save**.
2. Menu **Runtime → Run all**.
3. The first cell asks for your **Hugging Face token** (huggingface.co/settings/tokens, type *Write*). Paste it, press **Enter**.
   It is only used to store your trained model privately in your account so your Mac can download it.
4. Wait about 40–60 minutes with this tab open. You get the results table, and your model is saved.

Then on your Mac: `cd ~/aceh_code && git pull && ./run_app.sh` opens the app at http://127.0.0.1:7860.
It downloads your model once; after that it starts in seconds, offline. No need to train again.
If something goes wrong, copy the red error text and send it to Claude.

In [ ]:
# ---- settings ----------------------------------------------------------------------------------
from getpass import getpass
HF_TOKEN = getpass('Paste your Hugging Face WRITE token, then press Enter (Enter alone = download the model as a zip instead): ').strip()
DOWNLOAD_ZIP = False   # True: also download the trained model to this computer as a zip (about 1.2 GB)
USE_QURAN = False      # extra training data (Acehnese Quran translation): only after Andrie agrees
EPOCHS = 10            # maximum; training stops earlier when it stops improving

In [ ]:
# ---- code + packages + GPU check ---------------------------------------------------------------
import os
REPO = os.path.abspath('aceh_code')
if not os.path.isdir(REPO):
    !git clone -q -b claude/dataset-product-build-108xu8 https://github.com/Mutaib-yye/aceh_code {REPO}
%cd {REPO}
!git pull -q
!pip install -q "transformers==5.19.0" "sentencepiece>=0.2" "gradio==6.29.1" pymupdf pysbd datasketch rapidfuzz sacrebleu
import torch
if not torch.cuda.is_available():
    raise RuntimeError('No GPU. Menu: Runtime > Change runtime type > T4 GPU > Save. Then Runtime > Run all again.')
print('GPU:', torch.cuda.get_device_name(0))

In [ ]:
# ---- training data: NusaX + FLORES-200 + Hikayat Abu Sammah -> data/final/combined (leak-checked) ----
!python -m acehid fetch-nusax
if not os.path.exists('data/final/flores/dev.json'):
    !wget -q -O /tmp/flores200_dataset.tar.gz https://dl.fbaipublicfiles.com/nllb/flores200_dataset.tar.gz && python -m acehid flores /tmp/flores200_dataset.tar.gz || echo 'FLORES download failed: training without it (still fine)'
extra = '--quran' if USE_QURAN else ''
if USE_QURAN:
    !python -m acehid fetch-quran
!python -m acehid build-train {extra}
import json
m = json.load(open('data/final/combined/manifest.json'))
print('training pairs:', m['train']['n'], m['train']['by_source'])
print('validation:', m['validation']['n'], '| test sets:', {k: v['n'] for k, v in m['tests'].items()})

In [ ]:
# ---- fine-tune NLLB-200 (about 20-40 minutes on a T4) ----------------------------------------------
!python -m acehid train --epochs {EPOCHS}
if not os.path.exists('models/ace-id-nllb/RESULTS.md'):
    print('First attempt did not finish (usually: not enough GPU memory). Retrying with a smaller batch...')
    !python -m acehid train --epochs {EPOCHS} --batch-size 4
TRAINED = os.path.exists('models/ace-id-nllb/RESULTS.md')
print('TRAINING FINISHED' if TRAINED else 'TRAINING FAILED: copy the red error above and send it to Claude. The app will be published with the base model.')

In [ ]:
# ---- results: base model vs ours on test sentences it never saw ----------------------------------
from IPython.display import Markdown, display
if TRAINED:
    display(Markdown(open('models/ace-id-nllb/RESULTS.md', encoding='utf-8').read()))
else:
    print('No results table: training did not finish.')

In [ ]:
# ---- try it (change the sentences if you like) ----------------------------------------------------
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM
from acehid.normalize import normalize
MODEL_DIR = 'models/ace-id-nllb' if TRAINED else 'facebook/nllb-200-distilled-600M'
tok = AutoTokenizer.from_pretrained(MODEL_DIR, src_lang='ace_Latn')
model = AutoModelForSeq2SeqLM.from_pretrained(MODEL_DIR).to('cuda').eval()
def tr(text):
    x = tok([normalize(text)[0]], return_tensors='pt').to('cuda')
    with torch.no_grad():
        y = model.generate(**x, forced_bos_token_id=tok.convert_tokens_to_ids('ind_Latn'), num_beams=4, max_new_tokens=128)
    return tok.batch_decode(y, skip_special_tokens=True)[0]
for s in ['Lon jak u peukan uroe nyoe.', 'Teuma troih le Abu Sammah, budak indah paraih rupa']:
    print(s, '->', tr(s))

In [ ]:
# ---- save your trained model so the app on your Mac can use it (no training again later) ---------
import shutil
if not TRAINED:
    print('Training did not finish, so there is no model to save. Send the red error above to Claude.')
else:
    fp16 = 'models/ace-id-nllb-fp16'                    # half precision: 1.2 GB instead of 2.4 GB, same translations
    model.half().save_pretrained(fp16); tok.save_pretrained(fp16)
    for f in ('training_log.json', 'RESULTS.md', 'README.md'):
        shutil.copy(f'models/ace-id-nllb/{f}', fp16)
    if HF_TOKEN:
        from huggingface_hub import HfApi
        api = HfApi(token=HF_TOKEN)
        try:
            model_repo = f"{api.whoami()['name']}/ace-id-nllb"
        except Exception as e:
            raise RuntimeError('Hugging Face did not accept the token. Create a WRITE token at huggingface.co/settings/tokens, run the first cell again, then this cell.') from e
        api.create_repo(model_repo, private=True, exist_ok=True)
        api.update_repo_settings(model_repo, private=True)
        api.upload_folder(repo_id=model_repo, folder_path=fp16, commit_message='Fine-tuned Acehnese -> Indonesian NLLB')
        print('MODEL SAVED (private):', f'https://huggingface.co/{model_repo}')
        print('On your Mac:  cd ~/aceh_code && git pull && ./run_app.sh   (downloads it once, then opens the app)')
    if not HF_TOKEN or DOWNLOAD_ZIP:
        zip_path = shutil.make_archive(os.path.join(REPO, 'ace-id-nllb'), 'zip', fp16)
        print('zip ready:', zip_path, f'({os.path.getsize(zip_path) / 1e9:.1f} GB). Keep it in your Downloads folder; ./run_app.sh finds it there.')
        try:
            from google.colab import files
            files.download(zip_path)
        except ImportError:
            print('Not in Colab: copy the zip from', zip_path)

In [ ]:
# ---- optional: view the app right here (temporary public link while this tab is open) -------------
import sys, importlib.util, gc
del model; gc.collect(); torch.cuda.empty_cache()
sys.path.insert(0, os.path.join(REPO, 'space'))
spec = importlib.util.spec_from_file_location('space_app', os.path.join(REPO, 'space/app.py'))
app = importlib.util.module_from_spec(spec); spec.loader.exec_module(app)
app.start(MODEL_DIR, share=True)      # prints: Running on public URL: https://xxxx.gradio.live